##### Copyright 2019 The TensorFlow Authors.

In [3]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow 2 quickstart for beginners

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/quickstart/beginner"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This short introduction uses [Keras](https://www.tensorflow.org/guide/keras/overview) to:

1. Load a prebuilt dataset.
1. Build a neural network machine learning model that classifies images.
2. Train this neural network.
3. Evaluate the accuracy of the model.

This tutorial is a [Google Colaboratory](https://colab.research.google.com/notebooks/welcome.ipynb) notebook. Python programs are run directly in the browser—a great way to learn and use TensorFlow. To follow this tutorial, run the notebook in Google Colab by clicking the button at the top of this page.

1. In Colab, connect to a Python runtime: At the top-right of the menu bar, select *CONNECT*.
2. To run all the code in the notebook, select **Runtime** > **Run all**. To run the code cells one at a time, hover over each cell and select the **Run cell** icon.

![Run cell icon](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/quickstart/images/beginner/run_cell_icon.png?raw=1)

## Set up TensorFlow

Import TensorFlow into your program to get started:

In [4]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


If you are following along in your own development environment, rather than [Colab](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb), see the [install guide](https://www.tensorflow.org/install) for setting up TensorFlow for development.

Note: Make sure you have upgraded to the latest `pip` to install the TensorFlow 2 package if you are using your own development environment. See the [install guide](https://www.tensorflow.org/install) for details.

## Load a dataset

Load and prepare the MNIST dataset. The pixel values of the images range from 0 through 255. Scale these values to a range of 0 to 1 by dividing the values by `255.0`. This also converts the sample data from integers to floating-point numbers:

In [5]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Build a machine learning model

Build a `tf.keras.Sequential` model:

In [6]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) is useful for stacking layers where each layer has one input [tensor](https://www.tensorflow.org/guide/tensor) and one output tensor. Layers are functions with a known mathematical structure that can be reused and have trainable variables. Most TensorFlow models are composed of layers. This model uses the [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense), and [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) layers.

For each example, the model returns a vector of [logits](https://developers.google.com/machine-learning/glossary#logits) or [log-odds](https://developers.google.com/machine-learning/glossary#log-odds) scores, one for each class.

In [7]:
predictions = model(x_train[:1]).numpy()
predictions

array([[-0.21948618,  0.6410993 , -0.65421563, -0.5739754 ,  0.12064973,
         0.21663791, -0.019526  , -0.08230021, -0.72623336, -0.7419727 ]],
      dtype=float32)

The `tf.nn.softmax` function converts these logits to *probabilities* for each class:

In [8]:
tf.nn.softmax(predictions).numpy()

array([[0.08905318, 0.2105702 , 0.05765657, 0.06247362, 0.12513205,
        0.13773862, 0.10876546, 0.10214768, 0.05365027, 0.05281246]],
      dtype=float32)

Note: It is possible to bake the `tf.nn.softmax` function into the activation function for the last layer of the network. While this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output.

Define a loss function for training using `losses.SparseCategoricalCrossentropy`:

In [9]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

The loss function takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. This loss is equal to the negative log probability of the true class: The loss is zero if the model is sure of the correct class.

This untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to `-tf.math.log(1/10) ~= 2.3`.

In [10]:
loss_fn(y_train[:1], predictions).numpy()

np.float32(1.9823976)

Before you start training, configure and compile the model using Keras `Model.compile`. Set the [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) class to `adam`, set the `loss` to the `loss_fn` function you defined earlier, and specify a metric to be evaluated for the model by setting the `metrics` parameter to `accuracy`.

In [11]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## Train and evaluate your model

Use the `Model.fit` method to adjust your model parameters and minimize the loss:

In [12]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.8597 - loss: 0.4764
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9586 - loss: 0.1449
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9679 - loss: 0.1072
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9725 - loss: 0.0866
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9769 - loss: 0.0721


The `Model.evaluate` method checks the model's performance, usually on a [validation set](https://developers.google.com/machine-learning/glossary#validation-set) or [test set](https://developers.google.com/machine-learning/glossary#test-set).

In [13]:
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 1s - 3ms/step - accuracy: 0.9760 - loss: 0.0769


[0.07688713073730469, 0.9760000109672546]

The image classifier is now trained to ~98% accuracy on this dataset. To learn more, read the [TensorFlow tutorials](https://www.tensorflow.org/tutorials/).

If you want your model to return a probability, you can wrap the trained model, and attach the softmax to it:

In [14]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [15]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[6.6767605e-08, 5.1548067e-08, 4.5408701e-06, 1.2897039e-04,
        1.7551202e-11, 6.3044672e-07, 5.7267841e-12, 9.9983323e-01,
        3.6541081e-07, 3.2098214e-05],
       [5.8267169e-10, 7.8598445e-05, 9.9991596e-01, 4.9004243e-06,
        5.0799103e-15, 2.1652718e-07, 1.7692173e-08, 2.7558926e-13,
        3.2375576e-07, 2.0381777e-13],
       [1.7642858e-06, 9.9794978e-01, 1.6299388e-04, 1.2065287e-05,
        1.2595239e-04, 6.8501198e-05, 2.6103777e-05, 4.9213995e-04,
        1.1481900e-03, 1.2531420e-05],
       [9.9971837e-01, 1.2691621e-10, 3.1030995e-05, 3.9001673e-08,
        1.0524213e-06, 1.7190376e-06, 2.3221060e-04, 1.4277729e-06,
        8.6631388e-08, 1.3919685e-05],
       [5.6946469e-06, 3.1578747e-08, 2.9051532e-06, 1.4614617e-08,
        9.9917728e-01, 6.0866846e-06, 1.6386988e-05, 1.2600067e-04,
        4.8395559e-06, 6.6079159e-04]], dtype=float32)>

## Conclusion

Congratulations! You have trained a machine learning model using a prebuilt dataset using the [Keras](https://www.tensorflow.org/guide/keras/overview) API.

For more examples of using Keras, check out the [tutorials](https://www.tensorflow.org/tutorials/keras/). To learn more about building models with Keras, read the [guides](https://www.tensorflow.org/guide/keras). If you want learn more about loading and preparing data, see the tutorials on [image data loading](https://www.tensorflow.org/tutorials/load_data/images) or [CSV data loading](https://www.tensorflow.org/tutorials/load_data/csv).


In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from tensorflow.keras.datasets import mnist

# 加載 MNIST 數據集
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# 正規化數據
train_images = train_images.reshape((train_images.shape[0], 28, 28, 1))
test_images = test_images.reshape((test_images.shape[0], 28, 28, 1))
train_images, test_images = train_images / 255.0, test_images / 255.0

# 輸出訓練集和測試集的樣本數
print("Training data size:", train_images.shape[0])
print("Test data size:", test_images.shape[0])
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),  # 展平圖像數據
    layers.Dense(256, activation='tanh'),  # 第一層隱藏層，256個神經元，tanh激活
    layers.Dropout(0.0),  # 無 dropout
    layers.Dense(64),  # 第二層隱藏層，64個神經元，無激活函數
    layers.Dropout(0.5),  # Dropout機率 0.5
    layers.Dense(10, activation='softmax')  # 輸出層，10個類別，softmax激活
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


Training data size: 60000
Test data size: 10000


In [17]:
#訓練模型：
#訓練模型，最多 20 個 epoch。
model.fit(train_images, train_labels, epochs=20, batch_size=64)

Epoch 1/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.8203 - loss: 0.5800
Epoch 2/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9302 - loss: 0.2395
Epoch 3/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9549 - loss: 0.1498
Epoch 4/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9644 - loss: 0.1157
Epoch 5/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9732 - loss: 0.0911
Epoch 6/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9772 - loss: 0.0732
Epoch 7/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9812 - loss: 0.0591
Epoch 8/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9823 - loss: 0.0520
Epoch 9/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9848 - loss: 0.0464
Epoch 10/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9880 - loss: 0.0352
Epoch 11/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9892 - loss: 0.0328
Epoch 12/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step

In [18]:
#測試模型並打印準確度：
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("Test accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9763 - loss: 0.1191
Test accuracy: 0.980400025844574


In [19]:
#輸出測試數據的前 6 條樣本的預測結果：
predictions = model.predict(test_images[:6])

for i in range(6):
    print(f"Sample {i + 1}:")
    print(f"Predicted class: {np.argmax(predictions[i])}")
    print(f"Predicted probabilities: {predictions[i]}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Sample 1:
Predicted class: 7
Predicted probabilities: [1.21804894e-15 5.91069490e-16 7.78319358e-13 1.26294190e-12
 1.26413712e-16 1.22056405e-17 1.06007772e-22 1.00000000e+00
 2.57443836e-18 9.75785425e-11]
Sample 2:
Predicted class: 2
Predicted probabilities: [3.1149082e-08 7.5852436e-06 9.9999225e-01 3.2339731e-08 3.7161062e-13
 7.5642921e-12 1.4928064e-07 5.9806216e-11 1.5199509e-08 5.0463065e-14]
Sample 3:
Predicted class: 1
Predicted probabilities: [6.0746124e-13 1.0000000e+00 2.8532134e-09 9.8080192e-13 1.9911766e-09
 1.5080397e-11 5.8880401e-10 1.2985111e-09 1.6615868e-10 6.4962670e-14]
Sample 4:
Predicted class: 0
Predicted probabilities: [9.9997282e-01 1.6917570e-08 1.4391293e-07 3.2577988e-13 1.2984579e-10
 1.3300260e-10 2.7043006e-05 6.2498313e-09 1.0293828e-10 2.3872773e-10]
Sample 5:
Predicted class: 4
Predicted probabilities: [3.11817478e-17 4.71763531e-11 2.49233922e-14 1.00082746e-16
 9.99999881e-01 6.20002812e-16 2.50010465e-14 2